In [1]:
import sys
from pathlib import Path
BAMBOO_SETUP = Path.cwd()
sys.path.append(str((BAMBOO_SETUP/'src').resolve()))
import pandas as pd
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 1000)  # Set a larger width to fit the editor window
# pd.set_option('display.max_colwidth', None)  # Allow columns to be fully displayed
from post_processing.NN.DNNManager import DNNManager
from post_processing.NN.DNNModel import DNNModel

2024-10-17 21:53:59.399076: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-17 21:53:59.433669: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Welcome to JupyROOT 6.30/02


In [3]:
Z_OUTPUT_eos = Path('/eos/user/a/anunezde/Z_OUTPUT_eos')
workdir = Z_OUTPUT_eos / '2022_even_1013' / 'Reco'
sel_name = 'SL_res_2b_x'
total_inputs = 'input/vars40_new.txt'
DNNManagerdir =  'DNNManager_hm'

# Setup a DNN Manager to load all the data

In [4]:
manager = DNNManager(workdir=workdir, sel_name=sel_name, total_inputs=total_inputs, DNNManagerdir=DNNManagerdir)
manager.set_mode(mode = 'train_eval', rank_features=False)
total_df, _ = manager.start()

DNN Manager instantiated:
	WORKDIR:/eos/user/a/anunezde/Z_OUTPUT_eos/2022_even_1013/Reco
	RESULTSDIR:/eos/user/a/anunezde/Z_OUTPUT_eos/2022_even_1013/Reco/results
	DNNMANAGERDIR:/eos/user/a/anunezde/Z_OUTPUT_eos/2022_even_1013/Reco/DNNManager_hm
	Total inputs file: /afs/cern.ch/user/a/anunezde/bamboodev/hh/Bamboo_setup/src/post_processing/NN/input/vars40.txt
	Test models file: None

Loading data ...
Number of events read from DY_dl_mll_10to50: 135
Number of events read from DY_dl_mll_50_0J: 37
Number of events read from DY_dl_mll_50_1J: 442
Number of events read from DY_dl_mll_50_2J: 14155
Number of events read from bbWW_dl: 1457
Number of events read from bbWW_sl: 2931
Number of events read from bbtautau: 11089
Number of events read from WW: 593
Number of events read from WZ: 778
Number of events read from ZZ: 82
Number of events read from Wjets_0J: 17
Number of events read from Wjets_1J: 127
Number of events read from Wjets_2J: 6436
Number of events read from tWminus_dl: 30909
Number

# Set up your model config

In [6]:
from post_processing.NN.utils import ModelConfig
model_config = ModelConfig(
    name='multi_HH_ttbar_tW',
    type='multi',
    categorization={"HH": ["HH_bbWW"], "ttbar": ["ttbar"], "tW": ["tW"]},
    training_weight_sf={"HH_bbWW": 1.0, "ttbar": 8.0, "tW": 4.0},
    input_vars='All',
    architecture_in_yml=False,
    residual_network=True,
    hiddenlayers=[
        {"type": 'Dense', "units": 256, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4},
        {"type": 'Dense', "units": 256, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4},
        {"type": 'Dense', "units": 256, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4}
    ],
    outputlayers=[
        {"type": 'Dense', "units": 3, "kernel_initializer": 'normal', "activation": 'softmax', "act_regularizer": {'l2': 1e-4}, "name": 'output'}
    ],
    compiler={"optimizer": 'adam', "lr": 0.001, "loss": 'categorical_crossentropy'},
    fit={"batch_size": 1024, "epochs": 100, "validation_split": 0.25}
)

# DNNModel

In [5]:
DNN = DNNModel(model_config=model_config, modeldir= manager.DNNMANAGERDIR / 'model1')
model_df = DNN.set_model_df_from_total_df(total_df)
X_train, X_test, Y_train, Y_test, evs_test, sw_train = DNN.Full_Splitting(model_df)
input_layer, normalized_input = DNNModel.input_preprocessing(X_train)

Initializing model: multi_HH_ttbar_tW
Process_HH
Total sum of gen_Weights for HH: 384.61749267578125 
Total sum of sample_weights for HH: 2100511.25 
Process_ttbar
Total sum of gen_Weights for ttbar: 399578848.0 
Total sum of sample_weights for ttbar: 16804072.0 
Process_tW
Total sum of gen_Weights for tW: 8506427.0 
Total sum of sample_weights for tW: 8402045.0 
	Model dataframe:
            event  gen_Weight  nAK4  nAK4_btag     lep0_pt  lep0_eta  lep0_phi  ak4_jet0_pt  ak4_jet0_eta  ak4_jet0_phi  ak4_jet1_pt  ak4_jet1_eta  ak4_jet1_phi  ak4_jet2_pt  ak4_jet2_eta  ak4_jet2_phi  ak4_btag0_pt  ak4_btag0_eta  ak4_btag0_phi  ak4_btag1_pt  ak4_btag1_eta  ak4_btag1_phi      met_pt   met_phi   bjets_mbb  bjets_dPhi  bjets_dR  bjets_pt_bb  trijet_mInv   trijet_pt  trijet_pt_rat     blnu_mT     blnu_pt  trijet_bijet_dR  trijet_bijet_dPhi  bjet_bijet_dR  bjet_bijet_dPhi      all_pt     all_mInv  all_jets_HT  sample_weight  Class_HH  Class_ttbar  Class_tW
310419         18    3.805950     4    

In [2]:
Y_train_for_binary = Y_train['Class_ttbar']
Y_train_for_multi = Y_train
Y_test_for_binary = Y_test['Class_ttbar']
Y_test_for_multi = Y_test
Y_train = {'binary_output': Y_train_for_binary, 'multiclass_output': Y_train_for_multi}
Y_test = {'binary_output': Y_test_for_binary, 'multiclass_output': Y_test_for_multi}

NameError: name 'Y_train' is not defined

### Train and Evaluate

In [44]:
def train_and_eval(DNN, tf_model=None):
    if tf_model is not None:
        DNN.model = tf_model
    else:
        DNN.build_model(input_layer=input_layer, normalized_input=normalized_input)
    DNN.train_model(X_train, Y_train, sw_train)
    DNN.Evaluate(X_test, Y_test, evs_test)

In [19]:
from tensorflow.keras import layers, models, Input, regularizers
import tensorflow as tf
from post_processing.NN.model_builder import get_metrics
from tensorflow.keras.layers import Layer

class LessThanThreshold(Layer):
    def __init__(self, threshold=0.5, **kwargs):
        super(LessThanThreshold, self).__init__(**kwargs)
        self.threshold = threshold

    def call(self, inputs):
        return tf.cast(tf.less(inputs, self.threshold), tf.float32)

    def compute_mask(self, inputs, mask=None):
        # Pass the existing mask through, or create a new one if needed
        return mask

def hierarchical_model(input_layer, normalized_input):
    reg_l2 = regularizers.l2(1e-4)
    x = normalized_input
    x = layers.Dense(32, activation='relu', activity_regularizer=reg_l2)(x)
    x = layers.Dense(32, activation='relu', activity_regularizer=reg_l2)(x)
    x = layers.Dense(32, activation='relu', activity_regularizer=reg_l2)(x)
    x = layers.Dropout(0.4)(x)

    binary_output = layers.Dense(1, activation='sigmoid', kernel_initializer = 'normal', activity_regularizer=reg_l2, name='binary_output')(x)
    
    # Use the custom layer in your model
    mask = LessThanThreshold(threshold=0.5)(binary_output)
    masked_input = layers.Multiply()([normalized_input, mask])

    y = layers.Dense(32, activation='relu', activity_regularizer=reg_l2)(masked_input)
    y = layers.Dense(32, activation='relu', activity_regularizer=reg_l2)(y)
    y = layers.Dense(32, activation='relu', activity_regularizer=reg_l2)(y)
    y = layers.Dropout(0.4) (x)

    multiclass_output = layers.Dense(3, activation='softmax', kernel_initializer = 'normal', activity_regularizer=reg_l2, name='multiclass_output')(y)

    model = models.Model(inputs = input_layer, outputs=[binary_output, multiclass_output], name='hierarchical_model')

    model.compile(
        optimizer='adam',
        loss={'binary_output': 'binary_crossentropy', 'multiclass_output': 'categorical_crossentropy'},
        metrics = {'binary_output': get_metrics(classes=['ttbar']), 'multiclass_output': get_metrics(classes=DNN.classes)}
        )

    return model

model = hierarchical_model(input_layer, normalized_input)
DNN.model = model

In [20]:
DNN.train_model(X_train, Y_train, sw_train)

	Training model ...
Epoch 1/100
1231/1231 - 6s - 5ms/step - binary_output_accuracy: 1.0000 - binary_output_auc_pr: 0.9038 - binary_output_auc_roc: 0.6345 - binary_output_f1_score_macro: 0.9254 - binary_output_f1_score_micro: 0.9254 - binary_output_precision: 0.8826 - binary_output_recall: 0.9032 - loss: 18.9489 - multiclass_output_accuracy: 0.8351 - multiclass_output_auc_pr: 0.8465 - multiclass_output_auc_roc: 0.9304 - multiclass_output_f1_score_macro: 0.4059 - multiclass_output_f1_score_micro: 0.8351 - multiclass_output_precision: 0.8623 - multiclass_output_precision_HH: 0.1407 - multiclass_output_precision_tW: 0.3996 - multiclass_output_precision_ttbar: 0.8852 - multiclass_output_recall: 0.7145 - multiclass_output_recall_HH: 0.0345 - multiclass_output_recall_tW: 0.1107 - multiclass_output_recall_ttbar: 0.8123 - val_binary_output_accuracy: 1.0000 - val_binary_output_auc_pr: 0.9143 - val_binary_output_auc_roc: 0.6708 - val_binary_output_f1_score_macro: 0.9252 - val_binary_output_f1_sco

In [23]:
model_metrics = DNN.model.evaluate(X_test, Y_test, verbose=0, return_dict=True)   

In [25]:
print(model_metrics)

{'binary_output_accuracy': 1.0, 'binary_output_auc_pr': 0.9231382012367249, 'binary_output_auc_roc': 0.6985634565353394, 'binary_output_f1_score_macro': 0.9253435134887695, 'binary_output_f1_score_micro': 0.9253435134887695, 'binary_output_precision': 0.896233856678009, 'binary_output_recall': 0.889628529548645, 'loss': 1.0043025016784668, 'multiclass_output_accuracy': 0.8350904583930969, 'multiclass_output_auc_pr': 0.8653131127357483, 'multiclass_output_auc_roc': 0.9391947984695435, 'multiclass_output_f1_score_macro': 0.4789501130580902, 'multiclass_output_f1_score_micro': 0.8350903987884521, 'multiclass_output_precision': 0.8500707149505615, 'multiclass_output_precision_HH': 0.2066853791475296, 'multiclass_output_precision_tW': 0.42421528697013855, 'multiclass_output_precision_ttbar': 0.8952199816703796, 'multiclass_output_recall': 0.8042004108428955, 'multiclass_output_recall_HH': 0.34898751974105835, 'multiclass_output_recall_tW': 0.2449820637702942, 'multiclass_output_recall_ttbar

In [30]:
Y_pred_score = DNN.model.predict(X_test)

13129/13129 ━━━━━━━━━━━━━━━━━━━━ 4s 290us/step


In [34]:
binary_predictions = Y_pred_score[0]
multi_predicitions = Y_pred_score[1]

In [40]:
multi_predicitions.shape

(420103, 3)

In [55]:
for y_test, y_pred in zip(Y_test.values(),Y_pred_score):
    print(y_test.shape)
    print(y_pred.shape)

(420103,)
(420103, 1)
(420103, 3)
(420103, 3)
